# 测试客户端
测试客户端允许您使用该库对您的 ASGI 应用程序发出请求`httpx`。



In [ ]:
from starlette.responses import HTMLResponse
from starlette.testclient import TestClient


async def app(scope, receive, send):
    assert scope['type'] == 'http'
    response = HTMLResponse('<html><body>Hello, world!</body></html>')
    await response(scope, receive, send)


def test_app():
    client = TestClient(app)
    response = client.get('/')
    assert response.status_code == 200

测试客户端公开的接口与其他`httpx`会话相同。需要特别注意的是，发出请求的调用只是标准函数调用，而不是可等待函数。

您可以使用任何`httpx`标准 API，例如身份验证、会话 `cookie` 处理或文件上传。

例如，要在 `TestClient` 上设置标题，您可以执行以下操作：

In [ ]:
client = TestClient(app)

# Set headers on the client for future requests
client.headers = {"Authorization": "..."}
response = client.get("/")

# Set headers for each request separately
response = client.get("/", headers={"Authorization": "..."})

例如使用 `TestClient` 发送文件：

In [ ]:
client = TestClient(app)

# Send a single file
with open("example.txt", "rb") as f:
    response = client.post("/form", files={"file": f})

# Send multiple files
with open("example.txt", "rb") as f1:
    with open("example.png", "rb") as f2:
        files = {"file1": f1, "file2": ("filename", f2, "image/png")}
        response = client.post("/form", files=files)

欲了解更多信息，您可以查看`httpx` 文档。

默认情况下，`TestClient`会引发应用程序中发生的任何异常。有时您可能希望测试 500 错误响应的内容，而不是允许客户端引发服务器异常。在这种情况下，您应该使用`client = TestClient(app, raise_server_exceptions=False)`。

## 更改客户地址
默认情况下，`TestClient `将把客户端主机设置为 ，`"testserver"`并将端口设置为`50000`。

`client`您可以通过设置实例的属性来更改客户端地址`TestClient`：

In [ ]:
client = TestClient(app, client=('localhost', 8000))

## 选择异步后端
`TestClient`接受参数backend（字符串）和`backend_options`（字典）。这些选项将传递给`anyio.start_blocking_portal()`。 有关可接受的后端选项的更多信息，请参阅anyio 文档asyncio。默认情况下，将与默认选项一起使用。

要运行Trio，请传递backend="trio"。例如：

In [ ]:
def test_app()
    with TestClient(app, backend="trio") as client:
       ...

asyncio要使用运行uvloop，请传递backend_options={"use_uvloop": True}。例如：

In [ ]:
def test_app()
    with TestClient(app, backend_options={"use_uvloop": True}) as client:
       ...